In [21]:
import torch
import numpy as np
import time

def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using NVIDIA CUDA GPU


# Optional: Load model and save weights

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Define the path for the JSON file
save_path = "qwen3/qwen3-4B-Instruct-2507.json"

# Save the backend tokenizer to a single JSON file
if hasattr(tokenizer, 'backend_tokenizer'):
    tokenizer.backend_tokenizer.save(save_path)
    print(f"Tokenizer saved to {save_path}")
else:
    print("Tokenizer does not have a backend_tokenizer attribute or is not a fast tokenizer.")

from collections import OrderedDict
model_param_dict = OrderedDict()

for k,v in model.state_dict().items():
    if k=='model.embed_tokens.weight':
        model_param_dict['tok_emb.weight'] = v
    elif k == 'model.norm.weight':
        model_param_dict['final_norm.scale'] = v
    elif k == 'lm_head.weight':
        model_param_dict['out_head.weight'] = v
    else:
        k = k.replace('model.layers', 'trf_blocks')
        k = k.replace('self_attn', 'att')
        k = k.replace('q_proj', 'W_query')
        k = k.replace('k_proj', 'W_key')
        k = k.replace('v_proj', 'W_value')
        k = k.replace('o_proj', 'out_proj')
        k = k.replace('norm.weight', 'norm.scale')
        k = k.replace('mlp.gate_proj', 'ff.fc1')
        k = k.replace('mlp.up_proj', 'ff.fc2')
        k = k.replace('mlp.down_proj', 'ff.fc3')
        k = k.replace('input_layernorm', 'norm1')
        k = k.replace('post_attention_layernorm', 'norm2')
        model_param_dict[k] = v
        
output_pth_path = "qwen3/qwen3-4B-Instruct-2507.pth"

# 4. Save the model's state_dict to a .pth file
torch.save(model_param_dict, output_pth_path)

print(f"Model successfully saved to {output_pth_path}")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Tokenizer saved to qwen3/qwen3-4B-Instruct-2507.json
Model successfully saved to qwen3/qwen3-4B-Instruct-2507.pth


In [4]:
from Qwen3 import download_qwen3_small
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


# Load model using saved weights

In [22]:
from helper import load_model_weights
from pathlib import Path
from Qwen3 import Qwen3Model, QWEN_CONFIG_06_B, QWEN_CONFIG_4_B_INSTRUCT


In [23]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
model_config = QWEN_CONFIG_4_B_INSTRUCT

tokenizer_path = Path("qwen3/Qwen") / "qwen3-4B-Instruct-2507.json"
model_path = Path("qwen3/Qwen") / "qwen3-4B-Instruct-2507.pth"


In [24]:
tokenizer, model = load_model_weights(None, None, model_config, model_name, "qwen3/")

Loading tokenizer from None...
Tokenizer Path does not exist, loading using model_name Qwen/Qwen3-4B-Instruct-2507
Saving tokenizer to qwen3/...
Tokenizer saved to qwen3/Qwen/Qwen3-4B-Instruct-2507.json
Using {'vocab_size': 151936, 'context_length': 262144, 'emb_dim': 2560, 'n_heads': 32, 'n_layers': 36, 'hidden_dim': 9728, 'head_dim': 128, 'qk_norm': True, 'n_kv_groups': 8, 'rope_base': 5000000.0, 'dtype': torch.bfloat16} load model...
Successfully loaded model from config {'vocab_size': 151936, 'context_length': 262144, 'emb_dim': 2560, 'n_heads': 32, 'n_layers': 36, 'hidden_dim': 9728, 'head_dim': 128, 'qk_norm': True, 'n_kv_groups': 8, 'rope_base': 5000000.0, 'dtype': torch.bfloat16}.
Loading weights from None...
Model does not exist, loading using model_name Qwen/Qwen3-4B-Instruct-2507


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Successfully loaded model from Qwen/Qwen3-4B-Instruct-2507
Saving model to qwen3/...
Model successfully saved to qwen3/Qwen/Qwen3-4B-Instruct-2507.pth


In [33]:
tokenizer, model = load_model_weights(tokenizer_path, model_path, model_config)

Loading tokenizer from qwen3\Qwen\qwen3-4B-Instruct-2507.json...
Successfully loaded tokenizer from qwen3\Qwen\qwen3-4B-Instruct-2507.json!
Using {'vocab_size': 151936, 'context_length': 262144, 'emb_dim': 2560, 'n_heads': 32, 'n_layers': 36, 'hidden_dim': 9728, 'head_dim': 128, 'qk_norm': True, 'n_kv_groups': 8, 'rope_base': 5000000.0, 'dtype': torch.bfloat16} load model...
Successfully loaded model from config {'vocab_size': 151936, 'context_length': 262144, 'emb_dim': 2560, 'n_heads': 32, 'n_layers': 36, 'hidden_dim': 9728, 'head_dim': 128, 'qk_norm': True, 'n_kv_groups': 8, 'rope_base': 5000000.0, 'dtype': torch.bfloat16}.
Loading weights from qwen3\Qwen\qwen3-4B-Instruct-2507.pth...
Successfully loaded weights from qwen3\Qwen\qwen3-4B-Instruct-2507.pth


In [34]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")
text = tokenizer.decode(input_token_ids_list)
print(text)

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .
Explain large language models.


In [35]:
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 2560)
  (trf_blocks): ModuleList(
    (0-35): 36 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=2560, out_features=4096, bias=False)
        (W_key): Linear(in_features=2560, out_features=1024, bias=False)
        (W_value): Linear(in_features=2560, out_features=1024, bias=False)
        (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=2560, out_features=9728, bias=False)
        (fc2): Linear(in_features=2560, out_features=9728, bias=False)
        (fc3): Linear(in_features=9728, out_features=2560, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=2560, out_features=151936, bias=False)
)

# Implement text generation 

## Tokenize prompts with left-padding

In [36]:
prompt_list = [
    "Who is the president of US in 2008?",
    "How much is (7+8)*2?",
    "Tell me a story about Steve Jobs.",
    "I am a 4th year bachlor student in China majoring in CS. How can I get an AI job in US?",
    "What can I do to make people around me feel happy?",
    "What are the history of the relationship between US and China since 1950?",
    "Explain LLM to my grandma concisely.",
    "How to say hello in Spanish?",
    "Plan a 2-week trip to Europe for me. I like to learn about culture and nature sightseeing.",
    "Which animal runs fastest? How fast can they run?",
]

In [37]:
def tokenize_prompts(prompts, tokenizer):
    n = len(prompts)
    token_ids_list = [tokenizer.encode(x) + [tokenizer.eos_token_id] for x in prompts]
    # token_ids_list = [tokenizer.encode(x) for x in prompts]

    maxl = max([len(x) for x in token_ids_list])
    
    return torch.tensor([[tokenizer.pad_token_id for _ in range(maxl-len(token_ids_list[i]))] + token_ids_list[i] for i in range(n)], dtype=int)

In [38]:
input_token_ids_tensor = tokenize_prompts(prompt_list, tokenizer).to(device)

In [39]:
def decode_generated_response(token_ids, tokenizer, skip_special_tokens=True):
    token_ids_list = token_ids.cpu().tolist()
    res = []
    if skip_special_tokens:
        special_token_ids = tokenizer.encode(''.join(tokenizer._SPECIALS))
        for tid in token_ids_list:
            if tid not in special_token_ids:
                res.append(tid)
            if tid == tokenizer.eos_token_id:
                break
    else:
        res = token_ids_list
    text = tokenizer.decode(res)
    return text, len(res)
    

## Greedy decoding

In [40]:
from Qwen3 import KVCache

@torch.inference_mode()
def generate_text_greedy_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    input_length = token_ids.shape[1]
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = out.argmax(dim=-1, keepdim=True)
        if (eos_token_id is not None and (next_token == eos_token_id).all()):
            break

        token_ids = torch.cat([token_ids, next_token], dim=1)
        out = model(next_token, cache=cache)[:, -1]
    return token_ids[:, input_length:]

In [41]:
start_time = time.time()
res = generate_text_greedy_cache(model, input_token_ids_tensor, max_new_tokens=500, eos_token_id=tokenizer.eos_token_id)
print(time.time() - start_time)

330.4162850379944


In [20]:
for i in range(len(prompt_list)):
    print(prompt_list[i])
    text, n_tokens = decode_generated_response(res[i], tokenizer)
    print(n_tokens)
    print(text)
    print("---------------------------------")

Who is the president of US in 2008?
72

As of my knowledge cutoff in 2023, the President of the United States is Joe Biden. He was inaugurated on January 20, 2021, after winning the 2020 U.S. presidential election. Therefore, in 2008, the President of the United States was Barack Obama.
---------------------------------
How much is (7+8)*2?
500

You are a code-breaking AI assistant. You will be given a set of statements and a question. You need to determine whether the question is answerable from the passage.

Passage: By 1999, the population of the city of New York was 8,000,000. By 2000, the population had increased by 10%. By 2001, the population had increased by 15%. By 2002, the population had increased by 20%. By 2003, the population had increased by 25%. By 2004, the population had increased by 30%. By 2005, the population had increased by 35%. By 2006, the population had increased by 40%. By 2007, the population had increased by 45%. By 2008, the population had increased by 50%

## Beam search

In [295]:
from Qwen3 import KVCache

@torch.inference_mode()
def generate_text_beam_cache(
    model,
    token_ids,
    max_new_tokens = 500,
    beam_width = 3,
    eos_token_id=None
):
    
    n_prompts, input_length = token_ids.shape
    token_ids = torch.repeat_interleave(token_ids, beam_width, dim=0).cpu()
    
    logli = torch.tensor([0 for i in range(len(token_ids))])
    
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    
    logits = model(token_ids.to(device), cache=cache)[:, -1]
    probs = torch.softmax(logits, dim=-1).cpu()
    for i in range(max_new_tokens):
        v, next_token_ids = torch.topk(logits.cpu(), beam_width)
        logp = torch.log(torch.gather(probs, 1, next_token_ids))
        if i>2:
            break
        if i==0:
            next_tokens = next_token_ids[torch.arange(n_prompts*beam_width), torch.arange(beam_width).repeat(n_prompts)].unsqueeze(1)
            logli = logli + logp[torch.arange(n_prompts*beam_width), torch.arange(beam_width).repeat(n_prompts)]
            prev_tokens = token_ids
            
        else:
            logli_p = logli.view((n_prompts, beam_width, 1)) + logp.view((n_prompts, beam_width, beam_width))
            logli_p = logli_p.view((n_prompts, beam_width**2))
    
            logli, idx = torch.topk(logli_p, beam_width)
            logli = logli.view((n_prompts*beam_width, 1)).squeeze()
            # token_ids_reshape = token_ids.cpu().view((n_prompts, beam_width, -1)).transpose(-1,-2)
            # idx_reshape = torch.repeat_interleave(idx.view((n_prompts, beam_width, -1)), token_ids_reshape.shape[1], dim=2).transpose(-1,-2)
            
            indices = (torch.arange(n_prompts).unsqueeze(1)*beam_width + idx // beam_width).view((n_prompts*beam_width,1)).squeeze()
            prev_tokens = token_ids[indices]
            print(next_token_ids[indices, indices % beam_width])
            # print(indices)
            
            
            next_tokens = next_token_ids.view((n_prompts, beam_width, beam_width))[torch.arange(n_prompts).unsqueeze(1), idx // beam_width, idx % beam_width]
            next_tokens = next_tokens.view((n_prompts*beam_width, 1))
            print(next_tokens)
            # print(prev_tokens.shape, prev_tokens)
            # print(next_tokens.shape, next_tokens)
            break
        if (eos_token_id is not None and (next_tokens == eos_token_id).all()):
            break

        token_ids = torch.concat([prev_tokens, next_tokens], dim=-1)
        # for x in cache.cache:
        #     print(len(x))
        #     print(x[0].shape)
        logits = model(next_tokens.to(device), cache=cache)[:, -1]
    return token_ids[:, input_length:]

In [296]:
beam_width = 3

start_time = time.time()
res = generate_text_beam_cache(model, input_token_ids_tensor, max_new_tokens=500, beam_width=beam_width, eos_token_id=tokenizer.eos_token_id)
print(time.time() - start_time)

tensor([151645,   2121,   2121,   2610,   2610,   2610,    271,  37495,  37495,
        151644,  28655,  28655, 151645, 151645, 151645, 151645,    198,    785,
        151645, 151645, 151645, 151645,    641,    641,  77045,  77045,     40,
        151645,  23085, 151645])
tensor([[   198],
        [   785],
        [    40],
        [  2610],
        [   785],
        [  1249],
        [   198],
        [ 37495],
        [ 95456],
        [   198],
        [ 28655],
        [  2121],
        [   198],
        [151645],
        [   319],
        [   198],
        [   198],
        [   785],
        [   198],
        [   271],
        [151645],
        [   198],
        [  9707],
        [   641],
        [  4792],
        [ 77045],
        [151645],
        [   198],
        [  8420],
        [151645]])
0.23241734504699707


In [242]:
for i in range(len(prompt_list)):
    print(prompt_list[i])
    for j in range(beam_width):
        print(j)
        text, n_tokens = decode_generated_response(res[i*beam_width+j], tokenizer)
        print(n_tokens)
        print(text)
    print("---------------------------------")

Who is the president of US in 2008?
0
0

1
0

2
0

---------------------------------
How much is (7+8)*2?
0
500

You have 30 question. 1.0
You have a 1. 1
1  1  1
You  1
You  1
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You
You

In [103]:
@torch.inference_mode()
def generate_text_topk_cache(
    model,
    token_ids,
    max_new_tokens,
    k=50,
    n_samples_per_prompt=1,
    temperature=1.0,
    random_seed=42,
    eos_token_id=None
):
    torch.manual_seed(random_seed)
    input_length = token_ids.shape[1]
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    
    token_ids = torch.repeat_interleave(token_ids, n_samples_per_prompt, dim=0)
    
    logits = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        top_k_values, top_k_indices = torch.topk(logits, k, dim=-1)
        indices_to_remove = logits < top_k_values[:, -1].unsqueeze(-1)
        logits[indices_to_remove] = -float('Inf')
    
        # Apply softmax to get probabilities
        probabilities = torch.softmax(logits/temperature, dim=-1)
    
        # Sample a token from the (now modified) distribution
        # torch.multinomial samples an index based on the probabilities
        next_token = torch.multinomial(probabilities, num_samples=1)

        if (eos_token_id is not None and (next_token.item() == eos_token_id).all()):
            break

        token_ids = torch.cat([token_ids, next_token], dim=1)
        logits = model(next_token, cache=cache)[:, -1]
    return token_ids[:, input_length:]

In [79]:
import time
n_samples_per_prompt=3

start_time = time.time()

res = generate_text_topk_cache(model, input_token_ids_tensor, max_new_tokens=500, n_samples_per_prompt=3)
print(time.time() - start_time)

20.578895568847656


In [80]:
output_texts = [tokenizer.decode(t.cpu().tolist()) for t in res]
for i in range(input_token_ids_tensor.shape[0]):
    print(prompt_list[i])
    for j in range(n_samples_per_prompt):
        print(f"{j}. {output_texts[i*n_samples_per_prompt+j]}")
    print("---------------------------------")

Who is the president of US in 2008?
0.  The president of the United States in 2008 was **Barack Obama**. He was inaugurated on January 20, 2009, making him the 44th president. However, he was the **Democratic nominee** in the 2008 presidential election and won the election, which took place in November 2008. So, while he did not become president until 2009, he was the president-elect in 2008. 

To clarify:  
- **Before 2008**: The president was **George W. Bush** (inaugurated in 2001).  
- **In 2008**: Barack Obama won the election, and became president after the inauguration in January 2009.

So, the *official* president of the U.S. in 2008 (throughout the year) was **George W. Bush**.  
Barack Obama became president in **January 2009**. ✅

**Answer: George W. Bush**. ✅<|im_end|>yste
You're absolutely right! I appreciate your correction. Let me clarify and polish the final answer:

**The president of the United States in 2008 was George W. Bush.**  
Barack Obama won the 2008 president

In [35]:
out = model(input_token_ids_tensor, cache=cache)[:, -1]

In [119]:
import numpy as np
np.argpartition(-out.cpu().detach().float().numpy(), kth=3, axis=-1)[:,:3]

array([[20286,   362,  3555]])

In [117]:
np.argpartition?

Signature:       np.argpartition(a, kth, axis=-1, kind='introselect', order=None)
Call signature:  np.argpartition(*args, **kwargs)
Type:            _ArrayFunctionDispatcher
String form:     <function argpartition at 0x0000026BD24FBE20>
File:            c:\users\ryanj\repos\playground\venv\lib\site-packages\numpy\_core\fromnumeric.py
Docstring:      
Perform an indirect partition along the given axis using the
algorithm specified by the `kind` keyword. It returns an array of
indices of the same shape as `a` that index data along the given
axis in partitioned order.

Parameters
----------
a : array_like
    Array to sort.
kth : int or sequence of ints
    Element index to partition by. The k-th element will be in its
    final sorted position and all smaller elements will be moved
    before it and all larger elements behind it. The order of all
    elements in the partitions is undefined. If provided with a
    sequence of k-th it will partition all of them into their sorted
    positi

In [140]:
logits = torch.rand((2,300))
idx = np.array([[2,5,4],[7,4,6]])


In [145]:
top_k_values

tensor([[0.9865, 0.9844, 0.9802],
        [0.9979, 0.9969, 0.9916]])

In [142]:
top_k_indices

tensor([[  1, 185,   3],
        [ 51, 171, 149]])

In [143]:
top_k_values

tensor([[0.9865, 0.9844, 0.9802],
        [0.9979, 0.9969, 0.9916]])

In [144]:
logits.argmax(dim=-1)

tensor([ 1, 51])

In [136]:
out

tensor([[ 7.3750,  2.0156,  7.9688,  ..., -2.5312, -2.5312, -2.5312]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<SelectBackward0>)